# Master notebook — Nominal Rigidities in Online Prices (Colombia)
**Author:** David Mauricio Orozco Ríos (@Davoroz06) · Universidad ICESI / Dalhousie University

Runs the full pipeline end to end:

1. `10_build_datasets` – cleans the scraped files of each retailer (`11`–`14`) and joins them (`15`).
2. `20_estimations` – filters the panel (`21`), computes regular prices (Nakamura & Steinsson, `22`) and reference prices (Eichenbaum et al., `23`), then price-setting statistics and transition matrices (`24`).
3. `30_results` – builds the tables (`31`) and figures (`32`).

Step 2 and 3 are run once per specification reported in the paper.

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import papermill as pm

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text

## 1. Build the retailer datasets

In [ ]:
pm.execute_notebook("10_build_datasets.ipynb", None)

## 2–3. Estimations, tables and figures

In [ ]:
# Specifications reported in the paper:
#   case          regular_window  reference_window
#   "Comparison"  7               7                 -> main results (all retailers, same period)
#   "Comparison"  14              30                -> annex (bi-weekly / monthly windows)
#   "All"         7               7                 -> robustness (full collection period)
RUNS = [("Comparison", 7, 7), ("Comparison", 14, 30), ("All", 7, 7)]
run_filter = True  # set to False to reuse data/processed/Filter_Data_<case>.csv

In [ ]:
for case, regular_window, reference_window in RUNS:
    params = {"case": case, "regular_window": regular_window, "reference_window": reference_window}
    print("Processing", params)
    pm.execute_notebook("20_estimations.ipynb", None, parameters={**params, "run_filter": run_filter})
    pm.execute_notebook("30_results.ipynb", None, parameters=params)